# Task 4 — Join, Transformation Rules, and Testing

## SkyPrint

### Objective

Build a transformation pipeline that integrates validated flight, aircraft, and weather data for flights over Saudi Arabia.

The final dataset will support two main analytical goals:

1. **Air Traffic Analysis**
   - Preserve flight position and time information for traffic-density analysis.
   - Support future route-overload analysis when repeated OpenSky snapshots are collected.

2. **Environmental Analysis**
   - Enrich flight observations with aircraft metadata and weather conditions.
   - Prepare valid observations for fuel-consumption and emissions estimation using OpenAP.

### Design Principles

- Use validated outputs from Task 3.
- Keep the transformation compatible with future repeated snapshots.
- Document and test every join and transformation rule.
- Do not silently drop unmatched records.
- Check row counts and duplicate keys after joins.
- Produce the final analysis-ready dataset as `data/processed/final.csv`.

> Note: The current OpenSky data is a single snapshot. Therefore, the pipeline can prepare traffic features, but reliable route-overload analysis requires repeated snapshots over time.

## 1. Join Specification

Task 4 uses the validated outputs produced by Task 3.

### Input Datasets

| Dataset | Purpose | Key / Matching Method |
|---|---|---|
| Validated OpenSky | Base flight observations | `plane_id` |
| Validated Aircraft | Aircraft metadata and OpenAP type information | `plane_id` |
| Validated Weather | Atmospheric conditions | Spatial + temporal + altitude matching |

### Join 1 — OpenSky + Aircraft

- **Base dataset:** OpenSky
- **Join key:** `plane_id`
- **Join type:** Left join
- **Relationship:** Many-to-one
- **Reason:** Preserve every valid flight observation even when aircraft metadata is unavailable.
- **Duplicate handling:** Aircraft `plane_id` must be unique before the join. Duplicate keys are treated as an error rather than silently removed.

### Join 2 — Flight + Weather

Weather cannot be joined using `plane_id`.

Each flight observation will be matched to weather using:
- geographic location,
- observation time,
- and the atmospheric level appropriate to the aircraft altitude.

The matching logic will be implemented and tested explicitly before weather attributes are added.

### Row-count Policy

- Row counts will be measured before and after every join.
- Enrichment joins must not unexpectedly multiply flight observations.
- Unmatched records will be retained and flagged rather than silently dropped.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

# Project paths
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Interim folder:", INTERIM_DIR)
print("Processed folder:", PROCESSED_DIR)

Project root: c:\Users\W 10 Pro\Downloads\DE-SkyPrint-main\DE-SkyPrint-main
Interim folder: c:\Users\W 10 Pro\Downloads\DE-SkyPrint-main\DE-SkyPrint-main\data\interim
Processed folder: c:\Users\W 10 Pro\Downloads\DE-SkyPrint-main\DE-SkyPrint-main\data\processed


In [6]:
# Check Task 3 outputs available for Task 4

task3_files = sorted(INTERIM_DIR.glob("*.csv"))

print("Task 3 / interim files:")
for file in task3_files:
    print("-", file.name)

Task 3 / interim files:
- cleaned_aircraft.csv
- cleaned_opensky.csv
- cleaned_weather.csv
- rejected.csv
- rejected_aircraft.csv
- rejected_opensky.csv
- rejected_weather.csv
- schema_aircraft.csv
- schema_opensky.csv
- schema_weather.csv
- skyprint_readiness.csv
- validated.csv
- validated_aircraft.csv
- validated_opensky.csv
- validated_weather.csv
- validation_summary.csv


In [7]:
# Load validated Task 3 outputs

sky = pd.read_csv(INTERIM_DIR / "validated_opensky.csv")
aircraft = pd.read_csv(INTERIM_DIR / "validated_aircraft.csv")
weather = pd.read_csv(INTERIM_DIR / "validated_weather.csv")

print("Validated OpenSky:", sky.shape)
print("Validated Aircraft:", aircraft.shape)
print("Validated Weather:", weather.shape)

Validated OpenSky: (131, 22)
Validated Aircraft: (519969, 27)
Validated Weather: (2904, 34)


In [8]:
# Inspect columns and join keys before transformation

print("=== OPENSKY COLUMNS ===")
print(sky.columns.tolist())

print("\n=== AIRCRAFT COLUMNS ===")
print(aircraft.columns.tolist())

print("\n=== WEATHER COLUMNS ===")
print(weather.columns.tolist())

print("\n=== KEY CHECKS ===")
print("OpenSky rows:", len(sky))
print("OpenSky plane_id nulls:", sky["plane_id"].isna().sum())

print("Aircraft rows:", len(aircraft))
print("Aircraft plane_id nulls:", aircraft["plane_id"].isna().sum())
print("Aircraft plane_id duplicates:", aircraft["plane_id"].duplicated().sum())

print("Weather rows:", len(weather))
print("Weather location_id nulls:", weather["location_id"].isna().sum())

=== OPENSKY COLUMNS ===
['plane_id', 'flight_id', 'origin_country', 'time_position', 'last_contact', 'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity', 'true_track', 'vertical_rate', 'geo_altitude', 'source_type', 'category', 'rejection_reason', 'schema_valid', 'position_ready', 'trajectory_point_ready', 'kinematic_inputs_present', 'inside_extraction_bbox', 'inside_saudi_boundary']

=== AIRCRAFT COLUMNS ===
['plane_id', 'registration', 'manufacturericao', 'manufacturername', 'model', 'typecode', 'serialnumber', 'icaoaircrafttype', 'operator', 'operatoricao', 'owner', 'registered', 'reguntil', 'built', 'engines', 'categoryDescription', 'typecode_status', 'source_rows', 'typecode_candidates', 'rejection_reason', 'schema_valid', 'openap_typecode', 'openap_resolution', 'openap_supported', 'openap_resolution_note', 'metadata_ready', 'fuel_model_ready']

=== WEATHER COLUMNS ===
['observation_time', 'temperature_850hPa', 'wind_speed_850hPa', 'wind_direction_850hPa', 'geopotent

## 2. Join Flight and Aircraft Data

Join validated OpenSky flight observations with validated aircraft metadata using `plane_id`.

A left join is used to preserve all flight observations, including aircraft without matching metadata. The aircraft key must remain unique to prevent row multiplication.

In [9]:
# Join OpenSky flight observations with aircraft metadata

aircraft_cols = [
    "plane_id",
    "registration",
    "manufacturername",
    "model",
    "typecode",
    "engines",
    "categoryDescription",
    "openap_typecode",
    "openap_resolution",
    "openap_supported",
    "metadata_ready",
    "fuel_model_ready"
]

flight_aircraft = sky.merge(
    aircraft[aircraft_cols],
    on="plane_id",
    how="left",
    validate="many_to_one"
)

# Flag successful aircraft metadata matches
flight_aircraft["aircraft_metadata_matched"] = (
    flight_aircraft["metadata_ready"].fillna(False).astype(bool)
)

print("Rows before join:", len(sky))
print("Rows after join:", len(flight_aircraft))
print(
    "Aircraft metadata matched:",
    int(flight_aircraft["aircraft_metadata_matched"].sum())
)
print(
    "OpenAP ready:",
    int(flight_aircraft["openap_supported"].fillna(False).sum())
)

Rows before join: 131
Rows after join: 131
Aircraft metadata matched: 90
OpenAP ready: 84


In [10]:
# Test Join 1

assert len(flight_aircraft) == len(sky), \
    "Join 1 failed: row count changed."

assert flight_aircraft["plane_id"].isna().sum() == 0, \
    "Join 1 failed: null plane_id found."

assert aircraft["plane_id"].duplicated().sum() == 0, \
    "Join 1 failed: duplicate aircraft keys found."

print("JOIN 1 TEST: PASS")
print("All OpenSky observations were preserved without row multiplication.")

JOIN 1 TEST: PASS
All OpenSky observations were preserved without row multiplication.


## 3. Match Flight and Weather Data

Match each flight observation with the most appropriate weather observation using location and time.

Because weather conditions are provided at multiple atmospheric pressure levels, the aircraft altitude will then be used to select the weather level that best represents the aircraft's flight conditions.

The matching process must preserve all flight observations and avoid row multiplication.

In [11]:
# Inspect flight and weather matching fields

print("=== FLIGHT SAMPLE ===")
display(
    flight_aircraft[
        ["plane_id", "time_position", "latitude", "longitude",
         "baro_altitude", "geo_altitude"]
    ].head()
)

print("\n=== WEATHER SAMPLE ===")
display(
    weather[
        ["location_id", "observation_time", "latitude", "longitude"]
    ].head()
)

print("\nFlight time range:")
print(flight_aircraft["time_position"].min(), "->",
      flight_aircraft["time_position"].max())

print("\nWeather time range:")
print(weather["observation_time"].min(), "->",
      weather["observation_time"].max())

print("\nUnique weather locations:",
      weather["location_id"].nunique())

=== FLIGHT SAMPLE ===


,plane_id,time_position,latitude,longitude,baro_altitude,geo_altitude
0,739222,2026-09-18 15:20:35+00:00,32.8457,35.0522,426.72,449.58
1,74282d,2026-09-18 15:20:35+00:00,32.0513,35.0496,6019.80,6393.18
2,8015c2,2026-09-18 15:20:34+00:00,22.9726,51.5766,10668.00,11414.76
3,0180a0,2026-09-18 15:20:34+00:00,23.6820,50.2697,10972.80,11711.94
4,728679,2026-09-18 15:18:45+00:00,31.7282,36.0761,990.60,1059.18



=== WEATHER SAMPLE ===


,location_id,observation_time,latitude,longitude
0,0,2026-09-18 00:00:00+00:00,32.8125,35.125
1,0,2026-09-18 01:00:00+00:00,32.8125,35.125
2,0,2026-09-18 02:00:00+00:00,32.8125,35.125
3,0,2026-09-18 03:00:00+00:00,32.8125,35.125
4,0,2026-09-18 04:00:00+00:00,32.8125,35.125



Flight time range:
2026-09-18 15:16:09+00:00 -> 2026-09-18 15:20:36+00:00

Weather time range:
2026-09-18 00:00:00+00:00 -> 2026-09-18 23:00:00+00:00

Unique weather locations: 121


In [12]:
# Prepare timestamps for weather matching

flight_aircraft["time_position"] = pd.to_datetime(
    flight_aircraft["time_position"],
    utc=True
)

weather["observation_time"] = pd.to_datetime(
    weather["observation_time"],
    utc=True
)

# Match each flight observation to the nearest weather hour
flight_aircraft["weather_time"] = (
    flight_aircraft["time_position"].dt.round("h")
)

print(
    flight_aircraft[
        ["plane_id", "time_position", "weather_time"]
    ].head()
)

print(
    "\nWeather times available:",
    weather["observation_time"].nunique()
)

  plane_id             time_position              weather_time
0   739222 2026-09-18 15:20:35+00:00 2026-09-18 15:00:00+00:00
1   74282d 2026-09-18 15:20:35+00:00 2026-09-18 15:00:00+00:00
2   8015c2 2026-09-18 15:20:34+00:00 2026-09-18 15:00:00+00:00
3   0180a0 2026-09-18 15:20:34+00:00 2026-09-18 15:00:00+00:00
4   728679 2026-09-18 15:18:45+00:00 2026-09-18 15:00:00+00:00

Weather times available: 24


In [13]:
# Find the nearest weather location for each flight observation

weather_locations = (
    weather[["location_id", "latitude", "longitude"]]
    .drop_duplicates("location_id")
    .reset_index(drop=True)
)

def find_nearest_weather_location(row):
    distances = (
        (weather_locations["latitude"] - row["latitude"]) ** 2
        + (weather_locations["longitude"] - row["longitude"]) ** 2
    )

    nearest_index = distances.idxmin()
    return weather_locations.loc[nearest_index, "location_id"]

flight_aircraft["weather_location_id"] = flight_aircraft.apply(
    find_nearest_weather_location,
    axis=1
)

print(
    flight_aircraft[
        ["plane_id", "latitude", "longitude", "weather_location_id"]
    ].head()
)

print(
    "\nFlights with weather location:",
    flight_aircraft["weather_location_id"].notna().sum(),
    "/",
    len(flight_aircraft)
)

  plane_id  latitude  longitude  weather_location_id
0   739222   32.8457    35.0522                    0
1   74282d   32.0513    35.0496                    1
2   8015c2   22.9726    51.5766                    2
3   0180a0   23.6820    50.2697                    3
4   728679   31.7282    36.0761                    4

Flights with weather location: 131 / 131


In [14]:
# Calculate distance to the matched weather location

weather_location_lookup = weather_locations.rename(
    columns={
        "latitude": "weather_latitude",
        "longitude": "weather_longitude"
    }
)

flight_aircraft = flight_aircraft.merge(
    weather_location_lookup,
    left_on="weather_location_id",
    right_on="location_id",
    how="left",
    validate="many_to_one"
)

# Haversine distance in kilometers
lat1 = np.radians(flight_aircraft["latitude"])
lon1 = np.radians(flight_aircraft["longitude"])
lat2 = np.radians(flight_aircraft["weather_latitude"])
lon2 = np.radians(flight_aircraft["weather_longitude"])

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

flight_aircraft["weather_distance_km"] = (
    6371 * 2 * np.arcsin(np.sqrt(a))
)

print(
    flight_aircraft[
        [
            "plane_id",
            "latitude",
            "longitude",
            "weather_latitude",
            "weather_longitude",
            "weather_distance_km"
        ]
    ].head()
)

print("\nMaximum distance (km):",
      round(flight_aircraft["weather_distance_km"].max(), 2))

print("Average distance (km):",
      round(flight_aircraft["weather_distance_km"].mean(), 2))

  plane_id  latitude  longitude  weather_latitude  weather_longitude  \
0   739222   32.8457    35.0522         32.812500          35.125000   
1   74282d   32.0513    35.0496         32.125000          35.000000   
2   8015c2   22.9726    51.5766         23.022846          51.630096   
3   0180a0   23.6820    50.2697         23.725834          50.274550   
4   728679   31.7282    36.0761         31.687500          36.125000   

   weather_distance_km  
0             7.739360  
1             9.433627  
2             7.822978  
3             4.899068  
4             6.471454  

Maximum distance (km): 11.46
Average distance (km): 5.24


In [15]:
# Join flight observations with weather by location and time

flight_weather = flight_aircraft.merge(
    weather,
    left_on=["weather_location_id", "weather_time"],
    right_on=["location_id", "observation_time"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_weather")
)

print("Rows before weather join:", len(flight_aircraft))
print("Rows after weather join:", len(flight_weather))

print(
    "Weather matched:",
    int(flight_weather["weather_ready"].fillna(False).sum()),
    "/",
    len(flight_weather)
)

Rows before weather join: 131
Rows after weather join: 131
Weather matched: 131 / 131


In [16]:
# Test weather join

assert len(flight_weather) == len(flight_aircraft), \
    "Weather join failed: row count changed."

assert flight_weather["observation_time"].notna().all(), \
    "Weather join failed: some flights have no weather observation."

assert flight_weather["weather_ready"].fillna(False).all(), \
    "Weather join failed: some matched weather records are not ready."

assert (
    flight_weather["weather_location_id"]
    == flight_weather["location_id"]
).all(), "Weather join failed: location mismatch."

assert (
    flight_weather["weather_time"]
    == flight_weather["observation_time"]
).all(), "Weather join failed: time mismatch."

print("WEATHER JOIN TEST: PASS")
print("All flight observations have one valid weather match.")

WEATHER JOIN TEST: PASS
All flight observations have one valid weather match.


In [17]:
# Select the atmospheric pressure level nearest to each airborne flight altitude

pressure_levels = [850, 700, 500, 300, 250, 200]

# Prefer geometric altitude, fall back to barometric altitude
flight_weather["flight_altitude_m"] = (
    flight_weather["geo_altitude"]
    .fillna(flight_weather["baro_altitude"])
)

height_difference = pd.DataFrame(
    {
        level: abs(
            flight_weather[f"geopotential_height_{level}hPa"].to_numpy()
            - flight_weather["flight_altitude_m"].to_numpy()
        )
        for level in pressure_levels
    },
    index=flight_weather.index
)

# Keep pressure level missing when altitude is unavailable
valid_altitude = flight_weather["flight_altitude_m"].notna()

flight_weather["pressure_level_hpa"] = pd.NA

flight_weather.loc[valid_altitude, "pressure_level_hpa"] = (
    height_difference.loc[valid_altitude].idxmin(axis=1)
)

flight_weather["pressure_level_hpa"] = (
    flight_weather["pressure_level_hpa"].astype("Int64")
)

print("Total flight observations:", len(flight_weather))
print("Altitude available:", int(valid_altitude.sum()))
print("Altitude unavailable:", int((~valid_altitude).sum()))

print("\nSelected pressure levels:")
print(
    flight_weather["pressure_level_hpa"]
    .value_counts(dropna=False)
    .sort_index()
)

Total flight observations: 131
Altitude available: 127
Altitude unavailable: 4

Selected pressure levels:
pressure_level_hpa
200     23
250     26
300     14
500     18
700     18
850     28
<NA>     4
Name: count, dtype: Int64


In [18]:
# Verify pressure-level selection against aircraft altitude

pressure_check = (
    flight_weather
    .dropna(subset=["pressure_level_hpa"])
    .groupby("pressure_level_hpa")["flight_altitude_m"]
    .agg(["count", "min", "mean", "max"])
    .round(1)
)

print("=== PRESSURE LEVEL vs FLIGHT ALTITUDE ===")
display(pressure_check)

assert pressure_check.index.isin([850, 700, 500, 300, 250, 200]).all()
assert pressure_check["count"].sum() == 127

print("\nPRESSURE LEVEL CHECK: PASS")

=== PRESSURE LEVEL vs FLIGHT ALTITUDE ===


,count,min,mean,max
pressure_level_hpa,,,,
200,23,11879.6,12399.1,13860.8
250,26,10332.7,11206.7,11711.9
300,14,7848.6,9026.4,10340.3
500,18,4655.8,6048.6,7795.3
700,18,2529.8,3426.9,4511.0
850,28,45.7,1199.9,2301.2



PRESSURE LEVEL CHECK: PASS


## 4. Select Altitude-Matched Weather Conditions

For each flight observation with a valid altitude, select temperature, wind speed, and wind direction from the atmospheric pressure level matched to the aircraft altitude.

Flights without a valid altitude are retained, but altitude-dependent weather attributes remain unavailable.

These derived weather attributes will later support wind-aware flight features and fuel/emissions estimation.

In [19]:
# Select weather variables from the pressure level matched to each flight altitude

flight_weather["flight_temperature_c"] = np.nan
flight_weather["flight_wind_speed_kmh"] = np.nan
flight_weather["flight_wind_direction_deg"] = np.nan

for level in pressure_levels:
    mask = flight_weather["pressure_level_hpa"].eq(level)

    flight_weather.loc[mask, "flight_temperature_c"] = (
        flight_weather.loc[mask, f"temperature_{level}hPa"]
    )

    flight_weather.loc[mask, "flight_wind_speed_kmh"] = (
        flight_weather.loc[mask, f"wind_speed_{level}hPa"]
    )

    flight_weather.loc[mask, "flight_wind_direction_deg"] = (
        flight_weather.loc[mask, f"wind_direction_{level}hPa"]
    )

print("=== ALTITUDE-MATCHED WEATHER ===")

display(
    flight_weather[
        [
            "plane_id",
            "flight_altitude_m",
            "pressure_level_hpa",
            "flight_temperature_c",
            "flight_wind_speed_kmh",
            "flight_wind_direction_deg"
        ]
    ].head(10)
)

print("\nWeather attributes available:")
print("Temperature:", flight_weather["flight_temperature_c"].notna().sum())
print("Wind speed:", flight_weather["flight_wind_speed_kmh"].notna().sum())
print("Wind direction:", flight_weather["flight_wind_direction_deg"].notna().sum())

=== ALTITUDE-MATCHED WEATHER ===


,plane_id,flight_altitude_m,pressure_level_hpa,flight_temperature_c,flight_wind_speed_kmh,flight_wind_direction_deg
0,739222,449.58,850,15.8,39.7,279.0
1,74282d,6393.18,500,-5.6,57.4,259.0
2,8015c2,11414.76,250,-40.0,11.1,167.0
3,0180a0,11711.94,250,-40.5,28.8,182.0
4,728679,1059.18,850,16.9,41.5,287.0
5,4521ac,NaN,<NA>,NaN,NaN,NaN
6,70203f,11711.94,250,-39.5,23.6,345.0
7,89605b,6263.64,500,-4.4,26.6,98.0
8,8960ae,45.72,850,29.9,19.9,261.0
9,500205,10652.76,250,-43.0,96.5,238.0



Weather attributes available:
Temperature: 127
Wind speed: 127
Wind direction: 127


In [20]:
# Test altitude-matched weather attributes

valid_weather = flight_weather["pressure_level_hpa"].notna()

assert flight_weather.loc[
    valid_weather, "flight_temperature_c"
].notna().all(), "Missing matched temperature."

assert flight_weather.loc[
    valid_weather, "flight_wind_speed_kmh"
].notna().all(), "Missing matched wind speed."

assert flight_weather.loc[
    valid_weather, "flight_wind_direction_deg"
].notna().all(), "Missing matched wind direction."

assert (
    flight_weather.loc[valid_weather, "flight_wind_speed_kmh"] >= 0
).all(), "Negative wind speed found."

assert flight_weather.loc[
    valid_weather, "flight_wind_direction_deg"
].between(0, 360).all(), "Invalid wind direction found."

print("ALTITUDE-MATCHED WEATHER TEST: PASS")
print("Valid altitude/weather observations:", int(valid_weather.sum()))
print("Rows retained without altitude:", int((~valid_weather).sum()))

ALTITUDE-MATCHED WEATHER TEST: PASS
Valid altitude/weather observations: 127
Rows retained without altitude: 4


## 5. Derive Wind Components and Airspeed Inputs

Use the altitude-matched wind conditions together with aircraft ground speed and true track to derive wind-aware flight features.

OpenSky `velocity` represents ground speed, so it will not be treated directly as true airspeed. Wind speed and direction will be used to derive the aircraft airspeed inputs required for later fuel-flow and emissions estimation.

Rows with insufficient kinematic or weather data will be retained and flagged rather than assigned artificial values.

In [21]:
# Check inputs required for wind-aware airspeed calculation

airspeed_input_cols = [
    "velocity",
    "true_track",
    "flight_wind_speed_kmh",
    "flight_wind_direction_deg"
]

print("=== AIRSPEED INPUT CHECK ===")

for col in airspeed_input_cols:
    print(
        f"{col}:",
        "available =", flight_weather[col].notna().sum(),
        "| missing =", flight_weather[col].isna().sum()
    )

airspeed_inputs_ready = flight_weather[airspeed_input_cols].notna().all(axis=1)

print("\nTotal observations:", len(flight_weather))
print("Airspeed inputs ready:", int(airspeed_inputs_ready.sum()))
print("Airspeed inputs unavailable:", int((~airspeed_inputs_ready).sum()))

print("\nRows with unavailable inputs:")
display(
    flight_weather.loc[
        ~airspeed_inputs_ready,
        ["plane_id", "flight_id"] + airspeed_input_cols
    ]
)

=== AIRSPEED INPUT CHECK ===
velocity: available = 131 | missing = 0
true_track: available = 131 | missing = 0
flight_wind_speed_kmh: available = 127 | missing = 4
flight_wind_direction_deg: available = 127 | missing = 4

Total observations: 131
Airspeed inputs ready: 127
Airspeed inputs unavailable: 4

Rows with unavailable inputs:


,plane_id,flight_id,velocity,true_track,flight_wind_speed_kmh,flight_wind_direction_deg
5,4521ac,CYF462,4.37,120.94,NaN,NaN
13,45211b,CYF457,5.40,210.94,NaN,NaN
85,39ceb1,TVF3459,1.03,19.69,NaN,NaN
120,4d2426,WZZ3VY,5.14,196.88,NaN,NaN


In [22]:
# Convert wind speed from km/h to m/s
flight_weather["flight_wind_speed_ms"] = (
    flight_weather["flight_wind_speed_kmh"] / 3.6
)

# Relative angle between the direction the wind comes FROM
# and the aircraft ground track
relative_wind_angle_rad = np.deg2rad(
    flight_weather["flight_wind_direction_deg"]
    - flight_weather["true_track"]
)

# Wind component along the aircraft track
# Positive = headwind, Negative = tailwind
flight_weather["headwind_component_ms"] = (
    flight_weather["flight_wind_speed_ms"]
    * np.cos(relative_wind_angle_rad)
)

# Crosswind magnitude
flight_weather["crosswind_component_ms"] = (
    flight_weather["flight_wind_speed_ms"]
    * np.sin(relative_wind_angle_rad)
)

print("=== WIND COMPONENTS SAMPLE ===")

display(
    flight_weather[
        [
            "plane_id",
            "velocity",
            "true_track",
            "flight_wind_speed_ms",
            "flight_wind_direction_deg",
            "headwind_component_ms",
            "crosswind_component_ms"
        ]
    ].head(10)
)

=== WIND COMPONENTS SAMPLE ===


,plane_id,velocity,true_track,flight_wind_speed_ms,flight_wind_direction_deg,headwind_component_ms,crosswind_component_ms
0,739222,43.76,17.80,11.027778,279.0,-1.687094,-10.897963
1,74282d,209.04,299.97,15.944444,259.0,12.038900,-10.454195
2,8015c2,232.84,99.80,3.083333,167.0,1.194840,2.842411
3,0180a0,248.48,292.26,8.000000,182.0,-2.770246,-7.505047
4,728679,74.17,260.02,11.527778,287.0,10.273151,5.229916
5,4521ac,4.37,120.94,NaN,NaN,NaN,NaN
6,70203f,251.47,277.17,6.555556,345.0,2.473778,6.070892
7,89605b,188.02,117.90,7.388889,98.0,6.947684,-2.515027
8,8960ae,78.43,300.77,5.527778,261.0,4.248753,-3.536160
9,500205,236.56,152.26,26.805556,238.0,1.991185,26.731498


In [23]:
# Test derived wind components

valid_wind = airspeed_inputs_ready

assert flight_weather.loc[
    valid_wind, "flight_wind_speed_ms"
].ge(0).all(), "Negative wind speed found."

assert (
    flight_weather.loc[valid_wind, "headwind_component_ms"].abs()
    <= flight_weather.loc[valid_wind, "flight_wind_speed_ms"] + 1e-9
).all(), "Headwind component exceeds total wind speed."

assert (
    flight_weather.loc[valid_wind, "crosswind_component_ms"].abs()
    <= flight_weather.loc[valid_wind, "flight_wind_speed_ms"] + 1e-9
).all(), "Crosswind component exceeds total wind speed."

assert flight_weather.loc[
    valid_wind,
    ["headwind_component_ms", "crosswind_component_ms"]
].notna().all().all(), "Missing wind components found."

print("WIND COMPONENT TEST: PASS")
print("Valid wind-component observations:", int(valid_wind.sum()))
print("Rows retained without wind components:", int((~valid_wind).sum()))

WIND COMPONENT TEST: PASS
Valid wind-component observations: 127
Rows retained without wind components: 4


In [24]:
# Derive airspeed from ground-speed and wind vectors

track_rad = np.deg2rad(flight_weather["true_track"])
wind_from_rad = np.deg2rad(flight_weather["flight_wind_direction_deg"])

# Ground-velocity vector: east and north components
ground_east_ms = flight_weather["velocity"] * np.sin(track_rad)
ground_north_ms = flight_weather["velocity"] * np.cos(track_rad)

# Meteorological wind direction tells where wind comes FROM.
# Convert it to the direction the wind moves TOWARD.
wind_east_ms = -flight_weather["flight_wind_speed_ms"] * np.sin(wind_from_rad)
wind_north_ms = -flight_weather["flight_wind_speed_ms"] * np.cos(wind_from_rad)

# Air velocity = ground velocity - wind velocity
air_east_ms = ground_east_ms - wind_east_ms
air_north_ms = ground_north_ms - wind_north_ms

flight_weather["derived_airspeed_ms"] = np.sqrt(
    air_east_ms**2 + air_north_ms**2
)

# Keep airspeed unavailable when required inputs are unavailable
flight_weather.loc[
    ~airspeed_inputs_ready, "derived_airspeed_ms"
] = np.nan

print("=== DERIVED AIRSPEED SAMPLE ===")

display(
    flight_weather[
        [
            "plane_id",
            "velocity",
            "flight_wind_speed_ms",
            "headwind_component_ms",
            "crosswind_component_ms",
            "derived_airspeed_ms"
        ]
    ].head(10)
)

print(
    "\nDerived airspeed available:",
    flight_weather["derived_airspeed_ms"].notna().sum(),
    "/",
    len(flight_weather)
)

=== DERIVED AIRSPEED SAMPLE ===


,plane_id,velocity,flight_wind_speed_ms,headwind_component_ms,crosswind_component_ms,derived_airspeed_ms
0,739222,43.76,11.027778,-1.687094,-10.897963,43.461420
1,74282d,209.04,15.944444,12.038900,-10.454195,221.325937
2,8015c2,232.84,3.083333,1.194840,2.842411,234.052100
3,0180a0,248.48,8.000000,-2.770246,-7.505047,245.824345
4,728679,74.17,11.527778,10.273151,5.229916,84.604952
5,4521ac,4.37,NaN,NaN,NaN,NaN
6,70203f,251.47,6.555556,2.473778,6.070892,254.016334
7,89605b,188.02,7.388889,6.947684,-2.515027,194.983905
8,8960ae,78.43,5.527778,4.248753,-3.536160,82.754339
9,500205,236.56,26.805556,1.991185,26.731498,240.044248



Derived airspeed available: 127 / 131


In [25]:
# Test derived airspeed

valid_airspeed = airspeed_inputs_ready

assert flight_weather.loc[
    valid_airspeed, "derived_airspeed_ms"
].notna().all(), "Missing derived airspeed for valid inputs."

assert (
    flight_weather.loc[valid_airspeed, "derived_airspeed_ms"] > 0
).all(), "Non-positive derived airspeed found."

assert flight_weather.loc[
    ~valid_airspeed, "derived_airspeed_ms"
].isna().all(), "Airspeed was created for rows with insufficient inputs."

print("DERIVED AIRSPEED TEST: PASS")
print("Derived airspeed observations:", int(valid_airspeed.sum()))
print("Rows retained without derived airspeed:", int((~valid_airspeed).sum()))

print(
    "Derived airspeed range (m/s):",
    round(flight_weather.loc[valid_airspeed, "derived_airspeed_ms"].min(), 2),
    "->",
    round(flight_weather.loc[valid_airspeed, "derived_airspeed_ms"].max(), 2)
)

DERIVED AIRSPEED TEST: PASS
Derived airspeed observations: 127
Rows retained without derived airspeed: 4
Derived airspeed range (m/s): 43.46 -> 266.18


## 6. Prepare OpenAP Fuel-Flow Inputs

Prepare the validated flight observations for fuel-flow estimation using OpenAP.

Only observations with supported aircraft types and the required flight-state inputs will be eligible for fuel-flow estimation. Rows that are not eligible will be retained and explicitly flagged rather than assigned estimated values.

In [26]:
# Check readiness for OpenAP fuel-flow estimation

fuel_input_cols = [
    "openap_typecode",
    "derived_airspeed_ms",
    "flight_altitude_m",
    "vertical_rate"
]

print("=== OPENAP FUEL-FLOW INPUT CHECK ===")

for col in fuel_input_cols:
    print(
        f"{col}:",
        "available =", flight_weather[col].notna().sum(),
        "| missing =", flight_weather[col].isna().sum()
    )

fuel_ready = (
    flight_weather["fuel_model_ready"].fillna(False).astype(bool)
    & flight_weather["openap_typecode"].notna()
    & flight_weather["derived_airspeed_ms"].notna()
    & flight_weather["flight_altitude_m"].notna()
    & flight_weather["vertical_rate"].notna()
)

flight_weather["fuel_flow_ready"] = fuel_ready

print("\nTotal observations:", len(flight_weather))
print("Fuel-flow ready:", int(fuel_ready.sum()))
print("Not fuel-flow ready:", int((~fuel_ready).sum()))

display(
    flight_weather[
        [
            "plane_id",
            "flight_id",
            "openap_typecode",
            "fuel_model_ready",
            "derived_airspeed_ms",
            "flight_altitude_m",
            "vertical_rate",
            "fuel_flow_ready"
        ]
    ].head(10)
)

=== OPENAP FUEL-FLOW INPUT CHECK ===
openap_typecode: available = 84 | missing = 47
derived_airspeed_ms: available = 127 | missing = 4
flight_altitude_m: available = 127 | missing = 4
vertical_rate: available = 127 | missing = 4

Total observations: 131
Fuel-flow ready: 81
Not fuel-flow ready: 50


,plane_id,flight_id,openap_typecode,fuel_model_ready,derived_airspeed_ms,flight_altitude_m,vertical_rate,fuel_flow_ready
0,739222,4XCDE,NaN,False,43.461420,449.58,3.25,False
1,74282d,JAV271,NaN,NaN,221.325937,6393.18,6.18,False
2,8015c2,IGO094,NaN,NaN,234.052100,11414.76,0.00,False
3,0180a0,BNL741,A320,True,245.824345,11711.94,0.00,True
4,728679,IAW163,NaN,NaN,84.604952,1059.18,-3.90,False
5,4521ac,CYF462,NaN,NaN,NaN,NaN,NaN,False
6,70203f,BBC349,B77W,True,254.016334,11711.94,0.00,True
7,89605b,ROJ003,B737,True,194.983905,6263.64,-5.53,True
8,8960ae,DUB2,B744,True,82.754339,45.72,-3.58,True
9,500205,MEA422,A21N,True,240.044248,10652.76,0.00,True


In [27]:
from openap import prop

# Convert flight-state data to OpenAP input units
flight_weather["tas_kts"] = flight_weather["derived_airspeed_ms"] * 1.943844
flight_weather["altitude_ft"] = flight_weather["flight_altitude_m"] * 3.28084
flight_weather["vertical_rate_fpm"] = flight_weather["vertical_rate"] * 196.850394

# Estimate aircraft mass as 85% of MTOW
flight_weather["estimated_mass_kg"] = np.nan

for idx in flight_weather.index[flight_weather["fuel_flow_ready"]]:
    typecode = flight_weather.at[idx, "openap_typecode"]
    aircraft = prop.aircraft(typecode, use_synonym=True)

    flight_weather.at[idx, "estimated_mass_kg"] = (
        aircraft["mtow"] * 0.85
    )

print("=== OPENAP INPUTS PREPARED ===")
print(
    "Mass estimates available:",
    flight_weather["estimated_mass_kg"].notna().sum()
)

display(
    flight_weather.loc[
        flight_weather["fuel_flow_ready"],
        [
            "plane_id",
            "openap_typecode",
            "estimated_mass_kg",
            "tas_kts",
            "altitude_ft",
            "vertical_rate_fpm"
        ]
    ].head(10)
)

C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\prop.py:68: UserWarning: Aircraft: using synonym b77w for b77l
  warnings.warn(


=== OPENAP INPUTS PREPARED ===
Mass estimates available: 81


C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\prop.py:68: UserWarning: Aircraft: using synonym c550 for c56x
  warnings.warn(
C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\prop.py:68: UserWarning: Aircraft: using synonym glf6 for gl5t
  warnings.warn(


,plane_id,openap_typecode,estimated_mass_kg,tas_kts,altitude_ft,vertical_rate_fpm
3,0180a0,A320,66300.0,477.844179,38425.001230,0.000000
6,70203f,B77W,298775.0,493.768127,38425.001230,0.000000
7,89605b,B737,59500.0,379.018295,20550.000658,-1088.582679
8,8960ae,B744,337280.0,160.861525,150.000005,-704.724411
9,500205,A21N,82450.0,466.608570,34950.001118,0.000000
10,896120,B77W,298775.0,499.471957,39475.001263,0.000000
11,896190,A388,476000.0,365.982899,12050.000386,-511.811024
12,896179,B77L,298775.0,408.126661,26625.000852,0.000000
15,896264,B772,252450.0,484.483910,41575.001330,0.000000
16,471e16,B738,67150.0,368.799786,14650.000469,1728.346459


In [28]:
from openap import FuelFlow

flight_weather["fuel_flow_kg_s"] = np.nan

for idx in flight_weather.index[flight_weather["fuel_flow_ready"]]:
    typecode = flight_weather.at[idx, "openap_typecode"]

    ff = FuelFlow(
        ac=typecode,
        use_synonym=True
    )

    fuel_flow = ff.enroute(
        mass=flight_weather.at[idx, "estimated_mass_kg"],
        tas=flight_weather.at[idx, "tas_kts"],
        alt=flight_weather.at[idx, "altitude_ft"],
        vs=flight_weather.at[idx, "vertical_rate_fpm"]
    )

    flight_weather.at[idx, "fuel_flow_kg_s"] = fuel_flow

print("=== FUEL FLOW RESULTS ===")
print(
    "Fuel-flow estimates:",
    flight_weather["fuel_flow_kg_s"].notna().sum()
)

display(
    flight_weather.loc[
        flight_weather["fuel_flow_ready"],
        [
            "plane_id",
            "openap_typecode",
            "estimated_mass_kg",
            "tas_kts",
            "altitude_ft",
            "vertical_rate_fpm",
            "fuel_flow_kg_s"
        ]
    ].head(10)
)

C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\drag.py:72: UserWarning: Drag polar: using synonym a20n for a21n
  warnings.warn(
C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\prop.py:68: UserWarning: Aircraft: using synonym b77w for b77l
  warnings.warn(
C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\drag.py:72: UserWarning: Drag polar: using synonym b77w for b77l
  warnings.warn(
C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\prop.py:68: UserWarning: Aircraft: using synonym b77w for b77l
  warnings.warn(
C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\drag.py:72: UserWarning: Drag polar: using synonym b77w for b77l
  warnings.warn(
C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\drag.py:72: UserWarning: Drag polar: using synonym b38m for b39m
  warnings.warn(
C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\drag.py:

=== FUEL FLOW RESULTS ===
Fuel-flow estimates: 81


C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\prop.py:68: UserWarning: Aircraft: using synonym glf6 for gl5t
  warnings.warn(
C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\prop.py:68: UserWarning: Aircraft: using synonym glf6 for gl5t
  warnings.warn(
C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\drag.py:72: UserWarning: Drag polar: using synonym glf6 for gl5t
  warnings.warn(


,plane_id,openap_typecode,estimated_mass_kg,tas_kts,altitude_ft,vertical_rate_fpm,fuel_flow_kg_s
3,0180a0,A320,66300.0,477.844179,38425.001230,0.000000,0.750248
6,70203f,B77W,298775.0,493.768127,38425.001230,0.000000,3.259728
7,89605b,B737,59500.0,379.018295,20550.000658,-1088.582679,0.438160
8,8960ae,B744,337280.0,160.861525,150.000005,-704.724411,2.678186
9,500205,A21N,82450.0,466.608570,34950.001118,0.000000,0.552581
10,896120,B77W,298775.0,499.471957,39475.001263,0.000000,3.247328
11,896190,A388,476000.0,365.982899,12050.000386,-511.811024,3.407393
12,896179,B77L,298775.0,408.126661,26625.000852,0.000000,3.313083
15,896264,B772,252450.0,484.483910,41575.001330,0.000000,3.056819
16,471e16,B738,67150.0,368.799786,14650.000469,1728.346459,1.430612


In [29]:
# Test OpenAP fuel-flow results

fuel_results = flight_weather.loc[
    flight_weather["fuel_flow_ready"],
    "fuel_flow_kg_s"
]

assert len(fuel_results) == 81, "Unexpected number of fuel-flow-ready rows."
assert fuel_results.notna().all(), "Missing fuel-flow estimates found."
assert np.isfinite(fuel_results).all(), "Non-finite fuel-flow values found."
assert (fuel_results > 0).all(), "Non-positive fuel-flow values found."

print("FUEL FLOW TEST: PASS")
print("Fuel-flow observations:", len(fuel_results))
print(
    "Fuel-flow range (kg/s):",
    round(fuel_results.min(), 3),
    "->",
    round(fuel_results.max(), 3)
)
print(
    "Average fuel flow (kg/s):",
    round(fuel_results.mean(), 3)
)

FUEL FLOW TEST: PASS
Fuel-flow observations: 81
Fuel-flow range (kg/s): 0.133 -> 5.258
Average fuel flow (kg/s): 1.315


## 7. Estimate Aircraft Emissions

Estimate aircraft emission rates for observations with valid OpenAP fuel-flow results.

Emissions are calculated only for eligible observations using the aircraft type, fuel-flow estimate, true airspeed, and altitude. Rows without sufficient inputs are retained with unavailable emission estimates.

In [30]:
# Calculate aircraft emission rates using OpenAP

from openap import Emission

emission_cols = [
    "co2_g_s",
    "nox_g_s",
    "co_g_s",
    "hc_g_s",
    "sox_g_s",
    "soot_g_s"
]

# Initialize emission columns
for col in emission_cols:
    flight_weather[col] = np.nan

# Calculate emissions only for fuel-flow-ready observations
for idx in flight_weather.index[flight_weather["fuel_flow_ready"]]:
    typecode = flight_weather.at[idx, "openap_typecode"]

    emission = Emission(
        ac=typecode,
        use_synonym=True
    )

    ff = flight_weather.at[idx, "fuel_flow_kg_s"]
    tas = flight_weather.at[idx, "tas_kts"]
    alt = flight_weather.at[idx, "altitude_ft"]

    flight_weather.at[idx, "co2_g_s"] = emission.co2(ff)
    flight_weather.at[idx, "nox_g_s"] = emission.nox(ff, tas, alt)
    flight_weather.at[idx, "co_g_s"] = emission.co(ff, tas, alt)
    flight_weather.at[idx, "hc_g_s"] = emission.hc(ff, tas, alt)
    flight_weather.at[idx, "sox_g_s"] = emission.sox(ff)
    flight_weather.at[idx, "soot_g_s"] = emission.soot(ff)

print("=== EMISSION RESULTS ===")
print(
    "Emission estimates:",
    flight_weather["co2_g_s"].notna().sum()
)

display(
    flight_weather.loc[
        flight_weather["fuel_flow_ready"],
        [
            "plane_id",
            "openap_typecode",
            "fuel_flow_kg_s",
            "co2_g_s",
            "nox_g_s",
            "co_g_s",
            "hc_g_s",
            "sox_g_s",
            "soot_g_s"
        ]
    ].head(10)
)

C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\prop.py:68: UserWarning: Aircraft: using synonym b77w for b77l
  warnings.warn(


=== EMISSION RESULTS ===
Emission estimates: 81


C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\prop.py:68: UserWarning: Aircraft: using synonym c550 for c56x
  warnings.warn(
C:\Users\W 10 Pro\AppData\Roaming\Python\Python313\site-packages\openap\prop.py:68: UserWarning: Aircraft: using synonym glf6 for gl5t
  warnings.warn(


,plane_id,openap_typecode,fuel_flow_kg_s,co2_g_s,nox_g_s,co_g_s,hc_g_s,sox_g_s,soot_g_s
3,0180a0,A320,0.750248,2370.782893,10.431683,1.964537,0.176711,0.900297,0.022507
6,70203f,B77W,3.259728,10300.740203,82.073346,3.711571,0.197446,3.911673,0.097792
7,89605b,B737,0.438160,1384.585112,3.929463,3.295949,0.303632,0.525792,0.013145
8,8960ae,B744,2.678186,8463.067122,33.802980,5.624661,0.293006,3.213823,0.080346
9,500205,A21N,0.552581,1746.156092,5.823558,2.464007,0.029545,0.663097,0.016577
10,896120,B77W,3.247328,10261.555396,82.767817,2.973836,0.196556,3.896793,0.097420
11,896190,A388,3.407393,10767.362380,57.013934,4.254494,0.255959,4.088872,0.102222
12,896179,B77L,3.313083,10469.340871,79.912702,5.693735,0.182866,3.975699,0.099392
15,896264,B772,3.056819,9659.547444,104.164624,2.220291,0.150708,3.668183,0.091705
16,471e16,B738,1.430612,4520.735263,28.702914,1.425283,0.181409,1.716735,0.042918


In [31]:
# Test aircraft emission results

valid_emissions = flight_weather["fuel_flow_ready"]

assert flight_weather.loc[
    valid_emissions, "co2_g_s"
].notna().all(), "Missing CO2 estimates."

assert flight_weather.loc[
    valid_emissions, emission_cols
].ge(0).all().all(), "Negative emission values found."

assert flight_weather.loc[
    ~valid_emissions, emission_cols
].isna().all().all(), "Emissions were calculated for ineligible rows."

assert (
    flight_weather.loc[valid_emissions, emission_cols]
    .notna()
    .all()
    .all()
), "Missing emission estimates found."

print("EMISSION TEST: PASS")
print("Emission observations:", int(valid_emissions.sum()))
print("Rows retained without emissions:", int((~valid_emissions).sum()))

print(
    "CO2 range (g/s):",
    round(flight_weather.loc[valid_emissions, "co2_g_s"].min(), 3),
    "->",
    round(flight_weather.loc[valid_emissions, "co2_g_s"].max(), 3)
)

print(
    "NOx range (g/s):",
    round(flight_weather.loc[valid_emissions, "nox_g_s"].min(), 3),
    "->",
    round(flight_weather.loc[valid_emissions, "nox_g_s"].max(), 3)
)

EMISSION TEST: PASS
Emission observations: 81
Rows retained without emissions: 50
CO2 range (g/s): 420.901 -> 16615.761
NOx range (g/s): 0.607 -> 109.472


## 8. Prepare Final Observation-Level Dataset

Prepare an analysis-ready observation-level dataset that can be appended across repeated pipeline runs.

Each flight observation remains a separate record. The dataset preserves timestamp, position, aircraft, weather, fuel-flow, emissions, and Saudi-boundary information required for future spatiotemporal and traffic analysis.

No traffic corridor or route classification is created from the current single snapshot. These patterns will be derived after observations accumulate across repeated pipeline runs.

In [33]:
# Select analysis-ready observation-level columns

final_columns = [
    # Flight identity and time
    "plane_id",
    "flight_id",
    "origin_country",
    "time_position",
    "last_contact",

    # Position and movement
    "longitude",
    "latitude",
    "flight_altitude_m",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "inside_saudi_boundary",

    # Aircraft information
    "registration",
    "manufacturername",
    "model",
    "typecode",
    "openap_typecode",
    "aircraft_metadata_matched",

    # Matched weather
    "weather_time",
    "weather_location_id",
    "weather_latitude",
    "weather_longitude",
    "weather_distance_km",
    "pressure_level_hpa",
    "flight_temperature_c",
    "flight_wind_speed_kmh",
    "flight_wind_direction_deg",

    # Derived flight features
    "headwind_component_ms",
    "crosswind_component_ms",
    "derived_airspeed_ms",

    # Fuel estimation
    "fuel_flow_ready",
    "estimated_mass_kg",
    "fuel_flow_kg_s",

    # Emission estimates
    "co2_g_s",
    "nox_g_s",
    "co_g_s",
    "hc_g_s",
    "sox_g_s",
    "soot_g_s"
]

final_df = flight_weather[final_columns].copy()

print("=== FINAL DATASET STRUCTURE ===")
print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))
print("Duplicate column names:", final_df.columns.duplicated().sum())

display(final_df.head())

=== FINAL DATASET STRUCTURE ===
Rows: 131
Columns: 40
Duplicate column names: 0


,plane_id,flight_id,origin_country,time_position,last_contact,longitude,latitude,flight_altitude_m,on_ground,velocity,true_track,vertical_rate,inside_saudi_boundary,registration,manufacturername,model,typecode,openap_typecode,aircraft_metadata_matched,weather_time,weather_location_id,weather_latitude,weather_longitude,weather_distance_km,pressure_level_hpa,flight_temperature_c,flight_wind_speed_kmh,flight_wind_direction_deg,headwind_component_ms,crosswind_component_ms,derived_airspeed_ms,fuel_flow_ready,estimated_mass_kg,fuel_flow_kg_s,co2_g_s,nox_g_s,co_g_s,hc_g_s,sox_g_s,soot_g_s
0,739222,4XCDE,Israel,2026-09-18 15:20:35+00:00,2026-09-18 15:20:35+00:00,35.0522,32.8457,449.58,False,43.76,17.80,3.25,False,4X-CDE,NaN,NaN,C172,NaN,True,2026-09-18 15:00:00+00:00,0,32.812500,35.125000,7.739360,850,15.8,39.7,279.0,-1.687094,-10.897963,43.461420,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,74282d,JAV271,Jordan,2026-09-18 15:20:35+00:00,2026-09-18 15:20:35+00:00,35.0496,32.0513,6393.18,False,209.04,299.97,6.18,False,NaN,NaN,NaN,NaN,NaN,False,2026-09-18 15:00:00+00:00,1,32.125000,35.000000,9.433627,500,-5.6,57.4,259.0,12.038900,-10.454195,221.325937,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8015c2,IGO094,India,2026-09-18 15:20:34+00:00,2026-09-18 15:20:35+00:00,51.5766,22.9726,11414.76,False,232.84,99.80,0.00,True,NaN,NaN,NaN,NaN,NaN,False,2026-09-18 15:00:00+00:00,2,23.022846,51.630096,7.822978,250,-40.0,11.1,167.0,1.194840,2.842411,234.052100,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0180a0,BNL741,Libyan Arab Jamahiriya,2026-09-18 15:20:34+00:00,2026-09-18 15:20:35+00:00,50.2697,23.6820,11711.94,False,248.48,292.26,0.00,True,5A-BRB,NaN,NaN,A320,A320,True,2026-09-18 15:00:00+00:00,3,23.725834,50.274550,4.899068,250,-40.5,28.8,182.0,-2.770246,-7.505047,245.824345,True,66300.0,0.750248,2370.782893,10.431683,1.964537,0.176711,0.900297,0.022507
4,728679,IAW163,Iraq,2026-09-18 15:18:45+00:00,2026-09-18 15:18:45+00:00,36.0761,31.7282,1059.18,False,74.17,260.02,-3.90,False,NaN,NaN,NaN,NaN,NaN,False,2026-09-18 15:00:00+00:00,4,31.687500,36.125000,6.471454,850,16.9,41.5,287.0,10.273151,5.229916,84.604952,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
# Audit missing values in the final dataset

null_audit = pd.DataFrame({
    "missing_count": final_df.isna().sum(),
    "missing_percent": (final_df.isna().mean() * 100).round(1)
})

null_audit = null_audit[
    null_audit["missing_count"] > 0
].sort_values("missing_count", ascending=False)

print("=== FINAL DATASET NULL AUDIT ===")
display(null_audit)

=== FINAL DATASET NULL AUDIT ===


,missing_count,missing_percent
manufacturername,79,60.3
model,70,53.4
nox_g_s,50,38.2
co_g_s,50,38.2
hc_g_s,50,38.2
sox_g_s,50,38.2
soot_g_s,50,38.2
fuel_flow_kg_s,50,38.2
co2_g_s,50,38.2
estimated_mass_kg,50,38.2


In [35]:
# Check Saudi-boundary distribution

print("=== SAUDI BOUNDARY CHECK ===")
print(
    final_df["inside_saudi_boundary"]
    .value_counts(dropna=False)
)

print(
    "\nFuel/emission-ready observations inside Saudi boundary:",
    (
        final_df["inside_saudi_boundary"].fillna(False)
        & final_df["fuel_flow_ready"].fillna(False)
    ).sum()
)

=== SAUDI BOUNDARY CHECK ===
inside_saudi_boundary
False    103
True      28
Name: count, dtype: int64

Fuel/emission-ready observations inside Saudi boundary: 16


In [36]:
# Test final observation-level dataset

assert len(final_df) == len(flight_weather), \
    "Final dataset changed the observation grain."

assert final_df["plane_id"].notna().all(), \
    "Missing plane_id found."

assert final_df["time_position"].notna().all(), \
    "Missing observation timestamp found."

assert final_df["latitude"].notna().all(), \
    "Missing latitude found."

assert final_df["longitude"].notna().all(), \
    "Missing longitude found."

assert final_df["inside_saudi_boundary"].notna().all(), \
    "Missing Saudi-boundary flag found."

assert (
    final_df.loc[
        final_df["fuel_flow_ready"],
        "fuel_flow_kg_s"
    ].notna().all()
), "Fuel-ready observations contain missing fuel-flow estimates."

print("FINAL DATASET TEST: PASS")
print("Observations retained:", len(final_df))
print(
    "Inside Saudi boundary:",
    int(final_df["inside_saudi_boundary"].sum())
)
print(
    "Inside Saudi with emissions:",
    int(
        (
            final_df["inside_saudi_boundary"]
            & final_df["fuel_flow_ready"]
        ).sum()
    )
)

FINAL DATASET TEST: PASS
Observations retained: 131
Inside Saudi boundary: 28
Inside Saudi with emissions: 16


## 9. Transformation Rules and Final Output

Document the transformation rules implemented in Task 4 and verify the final analysis-ready dataset before export.

The rules describe the joins, weather matching, derived flight features, fuel-flow estimation, emission estimation, and preservation of Saudi airspace membership.

Final quality checks are performed before exporting the observation-level dataset to `data/processed/final.csv`.

In [37]:
# Transformation rules implemented in Task 4

rules = pd.DataFrame([
    {
        "Rule ID": "R1",
        "Description": "Join flight observations with aircraft metadata using a left join.",
        "Input Column(s)": "plane_id",
        "Output Column": "aircraft metadata columns"
    },
    {
        "Rule ID": "R2",
        "Description": "Match each flight observation to the nearest available weather location and observation time.",
        "Input Column(s)": "latitude, longitude, time_position",
        "Output Column": "weather_location_id, weather_time, weather_distance_km"
    },
    {
        "Rule ID": "R3",
        "Description": "Select the atmospheric pressure level closest to the aircraft altitude.",
        "Input Column(s)": "flight_altitude_m, geopotential height columns",
        "Output Column": "pressure_level_hpa"
    },
    {
        "Rule ID": "R4",
        "Description": "Select temperature and wind conditions from the altitude-matched pressure level.",
        "Input Column(s)": "pressure_level_hpa, pressure-level weather columns",
        "Output Column": "flight_temperature_c, flight_wind_speed_kmh, flight_wind_direction_deg"
    },
    {
        "Rule ID": "R5",
        "Description": "Derive wind components and air-relative speed from ground velocity, track, and matched wind.",
        "Input Column(s)": "velocity, true_track, flight_wind_speed_kmh, flight_wind_direction_deg",
        "Output Column": "headwind_component_ms, crosswind_component_ms, derived_airspeed_ms"
    },
    {
        "Rule ID": "R6",
        "Description": "Prepare supported observations for OpenAP fuel-flow estimation and estimate aircraft mass from MTOW.",
        "Input Column(s)": "openap_typecode, derived_airspeed_ms, flight_altitude_m, vertical_rate",
        "Output Column": "fuel_flow_ready, estimated_mass_kg"
    },
    {
        "Rule ID": "R7",
        "Description": "Estimate instantaneous aircraft fuel-flow rate using OpenAP.",
        "Input Column(s)": "openap_typecode, estimated_mass_kg, tas_kts, altitude_ft, vertical_rate_fpm",
        "Output Column": "fuel_flow_kg_s"
    },
    {
        "Rule ID": "R8",
        "Description": "Estimate instantaneous aircraft emission rates using OpenAP.",
        "Input Column(s)": "openap_typecode, fuel_flow_kg_s, tas_kts, altitude_ft",
        "Output Column": "co2_g_s, nox_g_s, co_g_s, hc_g_s, sox_g_s, soot_g_s"
    },
    {
        "Rule ID": "R9",
        "Description": "Preserve exact Saudi-boundary membership for downstream Saudi airspace analysis.",
        "Input Column(s)": "inside_saudi_boundary",
        "Output Column": "inside_saudi_boundary"
    }
])

display(rules)

,Rule ID,Description,Input Column(s),Output Column
0,R1,Join flight observations with aircraft metadat...,plane_id,aircraft metadata columns
1,R2,Match each flight observation to the nearest a...,"latitude, longitude, time_position","weather_location_id, weather_time, weather_dis..."
2,R3,Select the atmospheric pressure level closest ...,"flight_altitude_m, geopotential height columns",pressure_level_hpa
3,R4,Select temperature and wind conditions from th...,"pressure_level_hpa, pressure-level weather col...","flight_temperature_c, flight_wind_speed_kmh, f..."
4,R5,Derive wind components and air-relative speed ...,"velocity, true_track, flight_wind_speed_kmh, f...","headwind_component_ms, crosswind_component_ms,..."
5,R6,Prepare supported observations for OpenAP fuel...,"openap_typecode, derived_airspeed_ms, flight_a...","fuel_flow_ready, estimated_mass_kg"
6,R7,Estimate instantaneous aircraft fuel-flow rate...,"openap_typecode, estimated_mass_kg, tas_kts, a...",fuel_flow_kg_s
7,R8,Estimate instantaneous aircraft emission rates...,"openap_typecode, fuel_flow_kg_s, tas_kts, alti...","co2_g_s, nox_g_s, co_g_s, hc_g_s, sox_g_s, soo..."
8,R9,Preserve exact Saudi-boundary membership for d...,inside_saudi_boundary,inside_saudi_boundary


In [38]:
# Edge-case tests required for Task 4

# 1. Empty input should remain empty
empty_input = final_df.iloc[0:0].copy()
assert empty_input.empty, "Empty-input test failed."

# 2. Core observation keys must not be null
assert final_df["plane_id"].notna().all(), \
    "Null plane_id found."

assert final_df["time_position"].notna().all(), \
    "Null time_position found."

# 3. Missing flight_id is allowed and must not remove the observation
assert len(final_df) == 131, \
    "Observations were unexpectedly removed."

# 4. Ineligible fuel rows must retain missing estimates
not_fuel_ready = ~final_df["fuel_flow_ready"]

assert final_df.loc[
    not_fuel_ready, "fuel_flow_kg_s"
].isna().all(), \
    "Fuel flow found for an ineligible observation."

print("EDGE-CASE TESTS: PASS")
print("Empty input handled correctly.")
print("Null core keys checked.")
print("Optional missing flight_id retained.")
print("Ineligible fuel rows handled correctly.")

EDGE-CASE TESTS: PASS
Empty input handled correctly.
Null core keys checked.
Optional missing flight_id retained.
Ineligible fuel rows handled correctly.


In [39]:
# Final quality checks before export

assert len(final_df) == 131, \
    "Unexpected final row count."

assert final_df.columns.duplicated().sum() == 0, \
    "Duplicate column names found."

assert final_df["plane_id"].notna().all(), \
    "Missing plane_id found."

assert final_df["time_position"].notna().all(), \
    "Missing time_position found."

assert final_df[["latitude", "longitude"]].notna().all().all(), \
    "Missing flight coordinates found."

assert final_df["inside_saudi_boundary"].notna().all(), \
    "Missing Saudi-boundary flag found."

# Emissions must exist only for eligible observations
emission_cols = [
    "co2_g_s", "nox_g_s", "co_g_s",
    "hc_g_s", "sox_g_s", "soot_g_s"
]

assert final_df.loc[
    final_df["fuel_flow_ready"], emission_cols
].notna().all().all(), \
    "Missing emissions found for eligible observations."

assert final_df.loc[
    ~final_df["fuel_flow_ready"], emission_cols
].isna().all().all(), \
    "Emissions found for ineligible observations."

print("FINAL QUALITY CHECKS: PASS")
print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))
print("Fuel/emission-ready:", int(final_df["fuel_flow_ready"].sum()))
print("Inside Saudi boundary:", int(final_df["inside_saudi_boundary"].sum()))

FINAL QUALITY CHECKS: PASS
Rows: 131
Columns: 40
Fuel/emission-ready: 81
Inside Saudi boundary: 28


In [40]:
# Export the final analysis-ready dataset

final_path = PROCESSED_DIR / "final.csv"

final_df.to_csv(
    final_path,
    index=False
)

print("=== FINAL DATASET EXPORTED ===")
print("Path:", final_path)
print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))

=== FINAL DATASET EXPORTED ===
Path: c:\Users\W 10 Pro\Downloads\DE-SkyPrint-main\DE-SkyPrint-main\data\processed\final.csv
Rows: 131
Columns: 40


In [41]:
# Verify the exported final dataset

saved_final = pd.read_csv(final_path)

assert len(saved_final) == len(final_df), \
    "Saved file row count does not match final_df."

assert list(saved_final.columns) == list(final_df.columns), \
    "Saved file columns do not match final_df."

assert saved_final["plane_id"].notna().all(), \
    "Saved file contains missing plane_id values."

assert saved_final["time_position"].notna().all(), \
    "Saved file contains missing time_position values."

print("FINAL EXPORT VERIFICATION: PASS")
print("Saved rows:", len(saved_final))
print("Saved columns:", len(saved_final.columns))
print("File:", final_path.name)

FINAL EXPORT VERIFICATION: PASS
Saved rows: 131
Saved columns: 40
File: final.csv


In [43]:
# Test left-join behavior with a null key

left_test = pd.DataFrame({
    "plane_id": ["ABC123", None],
    "observation": ["matched_row", "null_key_row"]
})

right_test = pd.DataFrame({
    "plane_id": ["ABC123"],
    "aircraft_type": ["TEST_TYPE"]
})

result_test = left_test.merge(
    right_test,
    on="plane_id",
    how="left",
    validate="many_to_one"
)

# The left join must preserve both observations
assert len(result_test) == 2, \
    "Left join unexpectedly removed an observation."

# The valid key must match
assert result_test.loc[
    result_test["plane_id"] == "ABC123",
    "aircraft_type"
].iloc[0] == "TEST_TYPE", \
    "Valid key did not match correctly."

# The null key must be retained without a false match
null_row = result_test[result_test["plane_id"].isna()]

assert len(null_row) == 1, \
    "Null-key observation was not retained."

assert null_row["aircraft_type"].isna().all(), \
    "Null key created an unexpected match."

print("NULL-KEY JOIN TEST: PASS")
print("Valid key matched correctly.")
print("Null-key observation retained without a false match.")

NULL-KEY JOIN TEST: PASS
Valid key matched correctly.
Null-key observation retained without a false match.
